# Logistic Regression with Tribuo

Tribuo is the production-minded one. Where Smile took raw `double[][]`, Tribuo
works in **typed** terms: a `Dataset<Label>` of `Example<Label>` objects, trained
by a `Trainer<Label>` into a `Model<Label>`. The generics are the point — a
classification trainer cannot accidentally consume a regression dataset; the
compiler stops you.

We covered Tribuo end to end in [post 002](../README.md); here it slots into the
same comparison as everyone else. The shared `.java` files below do all the data
work. This notebook only does the Tribuo-specific part: **adapt our feature table
into typed Examples, train, evaluate, and read the provenance.**

In [ ]:
%%loadFromPOM
<dependency>
    <groupId>com.fasterxml.jackson.dataformat</groupId>
    <artifactId>jackson-dataformat-csv</artifactId>
    <version>2.17.2</version>
</dependency>
<dependency>
    <groupId>org.tribuo</groupId>
    <artifactId>tribuo-classification-sgd</artifactId>
    <version>4.3.2</version>
</dependency>

## The shared pipeline

Identical to every other notebook in this post — loading, name canonicalization,
feature engineering, the chronological split, the fair metrics, and the fixture
printout all live once in `shared/`.

In [ ]:
%load shared/Match.java
%load shared/DataLoader.java
%load shared/FormerName.java
%load shared/TeamNames.java
%load shared/EloRating.java
%load shared/RecentForm.java
%load shared/FeatureRow.java
%load shared/FeatureEngineering.java
%load shared/TrainTestSplit.java
%load shared/Metrics.java
%load shared/Predictions.java

In [ ]:
var all = DataLoader.loadAll("/home/jovyan/data/results.csv");
var names = TeamNames.load("/home/jovyan/data/former_names.csv");
var fe = FeatureEngineering.build(all, names);
var split = TrainTestSplit.chronological(fe.played(), 0.8);

System.out.println("train: " + split.train().size() + "   test: " + split.test().size());
System.out.println("upcoming fixtures to predict: " + fe.upcoming().size());

## The Tribuo adapter

This is where Tribuo differs most from Smile. Instead of a feature matrix, we
build a `MutableDataset<Label>` of `ArrayExample<Label>` — each example pairs a
feature vector with a typed `Label` (`"1"` = home win, `"0"` = not). The feature
*names* travel with the values, which is what lets Tribuo report on features by
name later. That typing is the whole pitch: a `Dataset<Label>` simply cannot be
fed to a regression trainer — it won't compile.

The conversion below is the entire Tribuo adapter: one loop per split.

In [ ]:
// Explicit imports, not `import org.tribuo.*`. The wildcard collides with names
// in scope and makes JShell (the Java kernel's engine) drop our earlier `var`
// bindings like `split`. Naming the exact types we use keeps everything intact.
import org.tribuo.Dataset;
import org.tribuo.MutableDataset;
import org.tribuo.classification.Label;
import org.tribuo.classification.LabelFactory;
import org.tribuo.classification.sgd.linear.LogisticRegressionTrainer;
import org.tribuo.classification.evaluation.LabelEvaluator;
import org.tribuo.impl.ArrayExample;
import org.tribuo.provenance.SimpleDataSourceProvenance;

var labelFactory = new LabelFactory();
String[] featureNames = FeatureRow.featureNames();

// The Tribuo adapter, inline. Build a typed MutableDataset<Label> from our
// feature rows: each example is a feature vector + a Label ("1" home win, "0"
// not). A plain loop per split — that's the whole adapter.
var trainData = new MutableDataset<Label>(
        new SimpleDataSourceProvenance("football-train", labelFactory), labelFactory);
for (FeatureRow r : split.train()) {
    trainData.add(new ArrayExample<>(new Label(r.homeWin() ? "1" : "0"), featureNames, r.features()));
}

var testData = new MutableDataset<Label>(
        new SimpleDataSourceProvenance("football-test", labelFactory), labelFactory);
for (FeatureRow r : split.test()) {
    testData.add(new ArrayExample<>(new Label(r.homeWin() ? "1" : "0"), featureNames, r.features()));
}

System.out.println("train examples: " + trainData.size() + "   test examples: " + testData.size());
System.out.println("label domain:   " + trainData.getOutputInfo().getDomain());

## Train

`LogisticRegressionTrainer` is Tribuo's SGD-trained logistic regression. As in
Smile, training is one line — the ceremony was all in shaping the typed dataset.

In [ ]:
var model = new LogisticRegressionTrainer().train(trainData);

## Evaluate — twice

Tribuo ships its own evaluator, `LabelEvaluator`, which prints a full per-class
report. That's genuinely useful, but every library reports differently, so to
compare fairly we *also* push Tribuo's probabilities through the same shared
`Metrics` used by the other notebooks. We show both.

In [ ]:
// Tribuo's native evaluation report.
var tribuoEval = new LabelEvaluator().evaluate(model, testData);
System.out.println(tribuoEval.toString());

In [ ]:
// Same model, scored through our shared Metrics for an apples-to-apples number.
int[] testY = TrainTestSplit.toY(split.test());
double[] testProbs = new double[split.test().size()];
for (int i = 0; i < split.test().size(); i++) {
    var ex = new ArrayExample<>(new Label("1"), featureNames, split.test().get(i).features());
    testProbs[i] = model.predict(ex).getOutputScores().get("1").getScore();
}
var metrics = Metrics.from(testProbs, testY);
System.out.println("Tribuo logistic regression (shared Metrics)");
System.out.println(metrics);

## Provenance — Tribuo's distinctive feature

This is what sets Tribuo apart from the others here. Every trained model carries
a complete, automatic record of how it was made: the trainer, its
hyperparameters, the random seed, the library and Java versions, the timestamp.
Nobody configured this logging — the framework captured it because provenance is
a first-class concern. In a regulated setting, this is the difference between
"we think the model was trained like X" and a signed, reproducible record.

(On *interpreting* the coefficients: the same scaling and collinearity caveats
from the Smile notebook apply here — same features, same math. We don't repeat
the autopsy; the lesson is that accuracy and interpretability are separate
properties, whichever library you use.)

In [ ]:
var provenance = model.getProvenance();
System.out.println("Trained at:     " + provenance.getTrainingTime());
System.out.println("Tribuo version: " + provenance.getTribuoVersion());
System.out.println("Java version:   " + provenance.getJavaVersion());
System.out.println("OS:             " + provenance.getOS());
System.out.println();
System.out.println("Trainer provenance:");
System.out.println(provenance.getTrainerProvenance().toString());

## Predict the 2026 World Cup group stage

The payoff, same as every notebook: the 44 fixtures with no result in the data —
the 2026 World Cup group stage. We build a throwaway Example per fixture (the
label is a placeholder — Tribuo needs *some* output type, but it's ignored at
prediction time) and read out P(home win).

In [ ]:
double[] upcomingProbs = new double[fe.upcoming().size()];
for (int i = 0; i < fe.upcoming().size(); i++) {
    var ex = new ArrayExample<>(new Label("1"), featureNames, fe.upcoming().get(i).features());
    upcomingProbs[i] = model.predict(ex).getOutputScores().get("1").getScore();
}
Predictions.print(fe.upcoming(), upcomingProbs);